# 1-2. RAG Indexing용 PDF → Markdown 전처리

## 학습 목표

- PDF를 Markdown으로 변환한다.
- `text_cleaner.py`를 import해서 페이지 단위 전처리를 수행한다.
- 공백 라인만 제거하고 실제 줄 구조는 유지한다.
- Markdown table 구조를 손상하지 않는다.
- 페이지 단위 `Document`를 생성해 RAG indexing에 사용한다.

## 기준

- 데이터 경로: `../data`
- 출력 경로: `../data`
- 전처리 모듈: `./text_cleaner.py`
- 청크 단위: `1 page = 1 chunk`

## 0. 필요 라이브러리 설치

In [16]:
# uv 환경
# !uv add pymupdf4llm langchain-core pandas

# pip 환경
# %pip install pymupdf4llm langchain-core pandas

## 1. Import 및 경로 설정

In [17]:
from pathlib import Path
import importlib

import pymupdf4llm
import pandas as pd
from langchain_core.documents import Document

import text_cleaner
importlib.reload(text_cleaner)

PDF_NAME = "공직자_민원응대_핵심_매뉴얼.pdf"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../data")

PDF_PATH = DATA_DIR / PDF_NAME

RAW_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_raw.md"
CLEAN_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_cleaned.md"
CHUNK_MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_rag_page_chunks.md"
CHUNK_CSV_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_rag_page_chunks.csv"
QUALITY_CSV_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_preprocess_quality.csv"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

print("PDF_PATH:", PDF_PATH)
print("text_cleaner:", text_cleaner.__file__)
print("CLEAN_MD_PATH:", CLEAN_MD_PATH)

PDF_PATH: ..\data\공직자_민원응대_핵심_매뉴얼.pdf
text_cleaner: c:\sk-encoa\llm_workspace\6.modular_rag_manual\notebooks\text_cleaner.py
CLEAN_MD_PATH: ..\data\공직자_민원응대_핵심_매뉴얼_cleaned.md


## 2. 메타데이터 및 점검 함수 정의

In [18]:
def infer_section(page: int) -> str:
    """페이지 번호로 문서 내 섹션을 추론한다."""
    if page <= 4:
        return "민원응대 관련 기본원칙"
    if page <= 7:
        return "일반적인 민원 응대요령"
    if page <= 12:
        return "특이민원 응대요령"
    if page <= 14:
        return "민원담당자 회복 및 보호조치"
    if page <= 17:
        return "질의응답"
    return "참고자료"


def infer_topic(text: str) -> str:
    """키워드 기반으로 페이지의 대표 주제를 추론한다."""
    rules = [
        ("반복전화", ["반복 전화", "반복전화", "동일민원"]),
        ("장시간 통화", ["장시간", "권장시간", "20분", "15분"]),
        ("상급자 통화 요구", ["상급자", "기관장", "부서장"]),
        ("폭언", ["폭언", "욕설", "협박", "성희롱"]),
        ("폭행", ["폭행", "공무집행방해"]),
        ("물품 파손", ["파손", "집기", "공용물건"]),
        ("위험물 소지", ["위험물", "흉기", "119"]),
        ("보호조치", ["휴식", "휴가", "심리상담", "치료"]),
        ("법적 근거", ["형법", "경범죄", "스토킹", "적용법률"]),
    ]

    for topic, keywords in rules:
        if any(keyword in text for keyword in keywords):
            return topic

    return "일반 민원응대"


def extract_markdown_text(item) -> str:
    """pymupdf4llm의 page_chunks 반환값에서 Markdown 텍스트를 추출한다."""
    if isinstance(item, str):
        return item

    if isinstance(item, dict):
        return item.get("text") or item.get("markdown") or item.get("content") or ""

    return str(item)


def make_markdown_document(pages: list[str], title: str) -> str:
    """페이지별 Markdown을 하나의 Markdown 문서로 결합한다."""
    lines = [f"# {title}"]

    for page_no, text in enumerate(pages, start=1):
        lines.append(f"<!-- page: {page_no} -->")
        lines.extend(text.strip().splitlines())

    # 공백 라인만 제거한다.
    return "\\n".join(line.rstrip() for line in lines if line.strip()) + "\\n"


def count_blank_lines(text: str) -> int:
    """공백 라인 수를 계산한다."""
    return sum(1 for line in text.splitlines() if not line.strip())


def count_table_lines(text: str) -> int:
    """Markdown table row 수를 계산한다."""
    return sum(1 for line in text.splitlines() if line.strip().count("|") >= 2)


def count_heading_lines(text: str) -> int:
    """Markdown heading 수를 계산한다."""
    return sum(1 for line in text.splitlines() if line.strip().startswith("#"))


def count_dot_sequences(text: str) -> int:
    """점선성 문자 패턴 수를 계산한다."""
    import re
    return len(re.findall(r"\\.{3,}|…+", text))


def make_quality_record(page: int, raw: str, cleaned: str) -> dict:
    """페이지별 전처리 품질 점검 record를 만든다."""
    return {
        "page": page,
        "raw_chars": len(raw),
        "cleaned_chars": len(cleaned),
        "raw_lines": len(raw.splitlines()),
        "cleaned_lines": len(cleaned.splitlines()),
        "raw_blank_lines": count_blank_lines(raw),
        "cleaned_blank_lines": count_blank_lines(cleaned),
        "raw_dot_sequences": count_dot_sequences(raw),
        "cleaned_dot_sequences": count_dot_sequences(cleaned),
        "raw_table_lines": count_table_lines(raw),
        "cleaned_table_lines": count_table_lines(cleaned),
        "raw_heading_lines": count_heading_lines(raw),
        "cleaned_heading_lines": count_heading_lines(cleaned),
    }

## 3. PDF를 페이지별 Markdown으로 변환

In [19]:
md_result = pymupdf4llm.to_markdown(
    str(PDF_PATH),
    page_chunks=True
)

print(md_result)
print(type(md_result))

if isinstance(md_result, str):
    raw_page_texts = [md_result]
else:
    raw_page_texts = [extract_markdown_text(page) for page in md_result]

raw_page_texts = [text for text in raw_page_texts if text and text.strip()]

print('raw_page_texts:', type(raw_page_texts[0]))

raw_markdown = make_markdown_document(
    raw_page_texts,
    title=f"{PDF_PATH.stem} Raw Markdown"
)

RAW_MD_PATH.write_text(raw_markdown, encoding="utf-8")

print("페이지 수:", len(raw_page_texts))
print("raw Markdown 저장:", RAW_MD_PATH)

[defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x000002D69745A980>, {'metadata': {'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe InDesign CC 2014 (Windows)', 'producer': 'Adobe PDF Library 11.0', 'creationDate': "D:20251130194656+09'00'", 'modDate': "D:20251130194704+09'00'", 'trapped': '', 'encryption': None, 'file_path': '..\\data\\공직자_민원응대_핵심_매뉴얼.pdf', 'page_count': 20, 'page_number': 1}, 'toc_items': [], 'page_boxes': [{'index': 0, 'class': 'picture', 'bbox': (0, 0, 0, 0), 'pos': (0, 2)}, {'index': 1, 'class': 'picture', 'bbox': (0, 0, 0, 0), 'pos': (2, 4)}, {'index': 2, 'class': 'picture', 'bbox': (0, 0, 0, 0), 'pos': (4, 6)}, {'index': 3, 'class': 'text', 'bbox': (37, 32, 135, 43), 'pos': (6, 20)}, {'index': 4, 'class': 'text', 'bbox': (37, 51, 135, 59), 'pos': (20, 43)}, {'index': 5, 'class': 'picture', 'bbox': (256, 0, 419, 79), 'pos': (43, 45)}, {'index': 6, 'class': 'picture', 'bbox': (98, 88, 219, 163), 'pos': (4

## 4. Markdown 구조 보존 전처리

In [20]:
cleaned_page_texts = text_cleaner.clean_pages(raw_page_texts)

cleaned_markdown = make_markdown_document(
    cleaned_page_texts,
    title=f"{PDF_PATH.stem} Cleaned Markdown"
)

CLEAN_MD_PATH.write_text(cleaned_markdown, encoding="utf-8")

print("cleaned Markdown 저장:", CLEAN_MD_PATH)
print("cleaned 페이지 수:", len(cleaned_page_texts))
print("cleaned Markdown 공백 라인 수:", count_blank_lines(cleaned_markdown))

cleaned Markdown 저장: ..\data\공직자_민원응대_핵심_매뉴얼_cleaned.md
cleaned 페이지 수: 20
cleaned Markdown 공백 라인 수: 0


## 5. 페이지 단위 Document 생성

In [21]:
page_docs = []

for page_no, text in enumerate(cleaned_page_texts, start=1):
    page_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": PDF_PATH.name,
                "page": page_no,
                "section": infer_section(page_no),
                "topic": infer_topic(text),
                "chunk_type": "page",
                "table_line_count": count_table_lines(text),
                "heading_count": count_heading_lines(text),
                "blank_line_count": count_blank_lines(text),
                "char_count": len(text),
            }
        )
    )

chunks = page_docs

print("page_docs 수:", len(page_docs))
print("chunks 수:", len(chunks))
print(chunks[0].metadata)
print(chunks[0].page_content[:1000])

page_docs 수: 20
chunks 수: 20
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 1, 'section': '민원응대 관련 기본원칙', 'topic': '일반 민원응대', 'chunk_type': 'page', 'table_line_count': 0, 'heading_count': 0, 'blank_line_count': 2, 'char_count': 55}
발 간 등 록 번 호

11-1741000-100065-14

현장공무원을 위한 민원응대 핵심매뉴얼


## 6. 전처리 품질 점검

In [22]:
quality_records = [
    make_quality_record(page_no, raw, cleaned)
    for page_no, (raw, cleaned) in enumerate(zip(raw_page_texts, cleaned_page_texts), start=1)
]

quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(QUALITY_CSV_PATH, index=False, encoding="utf-8-sig")

quality_df[[
    "page",
    "raw_chars",
    "cleaned_chars",
    "raw_blank_lines",
    "cleaned_blank_lines",
    "raw_dot_sequences",
    "cleaned_dot_sequences",
    "raw_table_lines",
    "cleaned_table_lines",
    "raw_heading_lines",
    "cleaned_heading_lines",
]]

,page,raw_chars,cleaned_chars,raw_blank_lines,cleaned_blank_lines,raw_dot_sequences,cleaned_dot_sequences,raw_table_lines,cleaned_table_lines,raw_heading_lines,cleaned_heading_lines
0,1,76,55,19,2,0,0,0,0,0,0
1,2,971,678,264,3,0,0,24,22,1,1
2,3,69,65,2,1,0,0,0,0,1,1
3,4,970,942,26,21,0,0,0,0,5,5
4,5,83,69,4,1,0,0,4,3,1,1
5,6,778,759,19,14,0,0,0,0,1,1
6,7,1014,976,23,18,0,0,0,0,3,3
7,8,990,967,25,22,0,0,0,0,12,12
8,9,1954,1891,40,39,0,0,0,0,14,14
9,10,2526,2453,73,62,0,0,0,0,20,20


## 7. RAG page chunk 저장

In [23]:
chunk_lines = [f"# {PDF_PATH.stem} RAG Page Chunks"]

for doc in chunks:
    meta = doc.metadata

    chunk_lines.append(f"## Page {meta['page']} | {meta['section']} | {meta['topic']}")
    chunk_lines.append(f"<!-- source: {meta['source']} -->")
    chunk_lines.append(f"<!-- chunk_type: {meta['chunk_type']} -->")
    chunk_lines.append(f"<!-- table_line_count: {meta['table_line_count']} -->")
    chunk_lines.append(f"<!-- blank_line_count: {meta['blank_line_count']} -->")
    chunk_lines.append(f"<!-- char_count: {meta['char_count']} -->")
    chunk_lines.extend(doc.page_content.splitlines())

chunk_markdown = "\n".join(line.rstrip() for line in chunk_lines if line.strip()) + "\n"
CHUNK_MD_PATH.write_text(chunk_markdown, encoding="utf-8")

chunk_df = pd.DataFrame([
    {
        "page": doc.metadata["page"],
        "section": doc.metadata["section"],
        "topic": doc.metadata["topic"],
        "chunk_type": doc.metadata["chunk_type"],
        "table_line_count": doc.metadata["table_line_count"],
        "heading_count": doc.metadata["heading_count"],
        "blank_line_count": doc.metadata["blank_line_count"],
        "char_count": doc.metadata["char_count"],
        "content": doc.page_content,
    }
    for doc in chunks
])

chunk_df.to_csv(CHUNK_CSV_PATH, index=False, encoding="utf-8-sig")

print("chunk Markdown 저장:", CHUNK_MD_PATH)
print("chunk CSV 저장:", CHUNK_CSV_PATH)
print("chunk Markdown 공백 라인 수:", count_blank_lines(chunk_markdown))

chunk Markdown 저장: ..\data\공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md
chunk CSV 저장: ..\data\공직자_민원응대_핵심_매뉴얼_rag_page_chunks.csv
chunk Markdown 공백 라인 수: 0


## 8. 표 포함 chunk 확인

In [24]:
table_chunks = [
    doc for doc in chunks
    if doc.metadata["table_line_count"] > 0
]

print("표 포함 chunk 수:", len(table_chunks))

if table_chunks:
    sample = table_chunks[0]
    print(sample.metadata)
    print(sample.page_content[:2000])

표 포함 chunk 수: 6
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 2, 'section': '민원응대 관련 기본원칙', 'topic': '반복전화', 'chunk_type': 'page', 'table_line_count': 22, 'heading_count': 1, 'blank_line_count': 3, 'char_count': 678}
## CONTENTS

|민원응대 관련 기본원칙<br>**I**|5|
|일반적인 민원 응대요령<br>**II**|9|
|특이민원 응대요령<br>**III**|13|
|1. 공통사항|15|
|2. 전화응대|15|
|2-1 정당한 사유없는 반복전화|15|
|2-2 정당한 사유없는 장시간 통화|16|
|2-3 상급자(기관장 등) 통화 요구|16|
|2-4 폭언(욕설, 협박, 성희롱 등)|17|
|3. 대면응대|18|
|3-1 폭언(욕설, 협박, 성희롱 등)|18|
|3-2 폭행|19|
|3-3 집기 또는 물품 등을 파손하는 경우|20|
|3-4 위험물을 소지하여 신변을 위협하는 경우|21|

|4. 온라인 민원 및 문서 민원응대|22|
|4-1 폭언(욕설, 협박, 성희롱 등)|22|
|4-2 정당한 사유없는 반복민원|23|
|민원담당자 회복 및 보호조치<br>**IV**|25|
|질의응답<br>**V**|29|
|[참고1] 특이민원 의미와 유형|35|
|[참고2] 위법행위 유형별 적용 법률|36|
|[참고3] 우울증 또는 스트레스 자가진단법|37|

※ 본 매뉴얼의 민원 응대요령은 민원공무원이 업무 수행 시 참고할 수 있도록 가이드 라인을 제시한 것 입니다. 민원업무 현장에서 다양한 응대 상황이 발생할 수 있으므로 각급 행정기관에서는 본 매뉴얼의 내용을 기관별 상황에 맞게 적절히 활용하시기 바랍니다.


## 9. chunk 통계 확인

In [25]:
chunk_df[[
    "page",
    "section",
    "topic",
    "table_line_count",
    "heading_count",
    "blank_line_count",
    "char_count",
]]

,page,section,topic,table_line_count,heading_count,blank_line_count,char_count
0,1,민원응대 관련 기본원칙,일반 민원응대,0,0,2,55
1,2,민원응대 관련 기본원칙,반복전화,22,1,3,678
2,3,민원응대 관련 기본원칙,일반 민원응대,0,1,1,65
3,4,민원응대 관련 기본원칙,장시간 통화,0,5,21,942
4,5,일반적인 민원 응대요령,일반 민원응대,3,1,1,69
5,6,일반적인 민원 응대요령,상급자 통화 요구,0,1,14,759
6,7,일반적인 민원 응대요령,일반 민원응대,0,3,18,976
7,8,특이민원 응대요령,반복전화,0,12,22,967
8,9,특이민원 응대요령,장시간 통화,0,14,39,1891
9,10,특이민원 응대요령,상급자 통화 요구,0,20,62,2453
